<a href="https://colab.research.google.com/github/husthorng/Backpropagation_NN/blob/main/2026_digitalization_ipynb_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# DHT_LDR Google Colab
# ==========================================================

import numpy as np

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

import time
from datetime import datetime
from zoneinfo import ZoneInfo


# ==========================================================
# 台灣時區
# ==========================================================
TW = ZoneInfo("Asia/Taipei")


# ==========================================================
# Google 授權
# ==========================================================
creds, _ = default()
gc = gspread.authorize(creds)


# ==========================================================
# Google Sheet
# ==========================================================
worksheet2 = gc.open("DHT_LDR").worksheet("input")
worksheet  = gc.open("DHT_LDR").worksheet("ESP32")


# ==========================================================
# 狀態
#
# lst[0] = 溫度 >= 28°C
# lst[1] = 溫度 >= 28°C 持續 30 秒
# lst[2] = LDR <= 2050
# ==========================================================
lst = [0, 0, 0]


# ==========================================================
# 計時變數
# ==========================================================
dt1 = None
dt2 = None


# ==========================================================
# 主迴圈
# ==========================================================
while True:

    try:

        # ==================================================
        # 取得目前台灣時間
        # ==================================================
        dt1 = datetime.now(TW)


        # ==================================================
        # 讀取溫度
        # ==================================================
        temp_data = worksheet.get_values("C2")

        temperature = float(
            temp_data[0][0]
        )


        # ==================================================
        # 讀取 Google Sheet B2
        # 只作為紀錄，不再拿來計時
        # ==================================================
        timestart = worksheet.get_values("B2")[0][0]


        print("\n========================================")

        print(
            "現在時間:",
            dt1.strftime("%Y-%m-%d %H:%M:%S")
        )

        print(
            "Temperature:",
            temperature
        )

        print(
            "Time Start:",
            timestart
        )


        # ==================================================
        # 🔥 溫度第一次 >= 28°C
        # ==================================================
        if temperature >= 32 and lst[0] == 0:

            lst[0] = 1

            # ----------------------------------------------
            # 使用台灣時間開始計時
            # ----------------------------------------------
            dt2 = datetime.now(TW)

            print(
                "🔥 Temp Start:",
                dt2.strftime("%Y-%m-%d %H:%M:%S")
            )


        # ==================================================
        # 🔄 溫度 < 28°C
        # ==================================================
        if temperature < 32:

            if lst[0] == 1:

                print(
                    "🔄 Temperature < 28°C → Reset"
                )

            lst = [0, 0, 0]

            dt2 = None


        # ==================================================
        # ⏱ 溫度持續時間
        # ==================================================
        if lst[0] == 1 and dt2 is not None:

            # ----------------------------------------------
            # 目前台灣時間
            # ----------------------------------------------
            dt1 = datetime.now(TW)


            # ----------------------------------------------
            # 計算經過秒數
            # ----------------------------------------------
            diff = (
                dt1 - dt2
            ).total_seconds()


            print(
                "Dt1:",
                dt1.strftime("%Y-%m-%d %H:%M:%S")
            )

            print(
                "Dt2:",
                dt2.strftime("%Y-%m-%d %H:%M:%S")
            )

            print(
                f"⏱ Duration: {diff:.1f} sec"
            )


            # ----------------------------------------------
            # >= 30 秒
            # ----------------------------------------------
            if diff >= 30:

                lst[1] = 1

                print(
                    "✅ Temperature >= 28°C "
                    "持續 30 秒"
                )

            else:

                lst[1] = 0

                print(
                    f"⏳ 尚未達 30 秒"
                )


        # ==================================================
        # 📌 讀取 LDR
        # ==================================================
        photo_data = worksheet.get_values("E2")

        Photosensitive = float(
            photo_data[0][0]
        )


        # ==================================================
        # 💡 LDR 判斷
        #
        # <= 2050 → 1
        # > 2050  → 0
        # ==================================================
        if Photosensitive <= 2050:

            lst[2] = 1

        else:

            lst[2] = 0


        # ==================================================
        # 顯示 LDR
        # ==================================================
        print(
            f"💡 LDR: {Photosensitive:.1f}"
        )


        # ==================================================
        # 顯示狀態
        # ==================================================
        print(
            "📌 狀態:",
            lst
        )


        # ==================================================
        # 讀取 input 第一列
        # ==================================================
        rowi = worksheet2.row_values(1)


        if len(rowi) >= 3:

            try:

                row1 = np.array(
                    rowi[:3],
                    dtype=int
                ).tolist()

            except:

                row1 = [-1, -1, -1]

        else:

            row1 = [-1, -1, -1]


        print(
            "Sheet input:",
            row1
        )


        # ==================================================
        # 狀態改變 → 寫入 Google Sheet
        # ==================================================
        if row1 != lst:

            # ----------------------------------------------
            # 使用台灣完整日期時間
            # ----------------------------------------------
            current_datetime = datetime.now(TW).strftime(
                "%Y-%m-%d %H:%M:%S"
            )


            row_new = [
                lst[0],
                lst[1],
                lst[2],
                "",
                current_datetime
            ]


            # ----------------------------------------------
            # 插入第一列
            # ----------------------------------------------
            worksheet2.insert_row(
                row_new,
                1
            )


            print(
                "💾 New Record Saved!"
            )

            print(
                "   Data:",
                row_new
            )


    except Exception as e:

        print(
            "\n⚠️ Get Value Error, Continue..."
        )

        print(
            "錯誤:",
            e
        )


    # ======================================================
    # 每 2 秒執行一次
    # ======================================================
    time.sleep(2)


現在時間: 2026-09-03 15:16:00
Temperature: 31.3
Time Start: 下午 3:11:58
💡 LDR: 3212.0
📌 狀態: [0, 0, 0]
Sheet input: [1, 1, 0]
💾 New Record Saved!
   Data: [0, 0, 0, '', '2026-09-03 15:16:00']

現在時間: 2026-09-03 15:16:03
Temperature: 31.3
Time Start: 下午 3:11:58
💡 LDR: 3212.0
📌 狀態: [0, 0, 0]
Sheet input: [1, 1, 0]
💾 New Record Saved!
   Data: [0, 0, 0, '', '2026-09-03 15:16:03']

現在時間: 2026-09-03 15:16:05
Temperature: 31.3
Time Start: 下午 3:11:58
💡 LDR: 3212.0
📌 狀態: [0, 0, 0]
Sheet input: [1, 1, 0]
💾 New Record Saved!
   Data: [0, 0, 0, '', '2026-09-03 15:16:06']

現在時間: 2026-09-03 15:16:08
Temperature: 31.3
Time Start: 下午 3:11:58
💡 LDR: 3212.0
📌 狀態: [0, 0, 0]
Sheet input: [1, 1, 0]
💾 New Record Saved!
   Data: [0, 0, 0, '', '2026-09-03 15:16:09']

現在時間: 2026-09-03 15:16:11
Temperature: 31.3
Time Start: 下午 3:11:58
💡 LDR: 3212.0
📌 狀態: [0, 0, 0]
Sheet input: [1, 1, 0]
💾 New Record Saved!
   Data: [0, 0, 0, '', '2026-09-03 15:16:12']

現在時間: 2026-09-03 15:16:14
Temperature: 31.3
Time Start: 下午 3:11:

KeyboardInterrupt: 